In [ ]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

In [ ]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

bronze_stream = raw_stream.select(
    F.col("key").cast("string"),
    F.col("value").cast("string"),       # raw JSON string
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("kafka_timestamp"),
)

query = (
    bronze_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/bronze")
    .trigger(availableNow=True)          # process all available, then stop
    .toTable("lakehouse.taxi.bronze")
)
print("Done")

In [ ]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 3").show(truncate=False)

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Schema
trip_schema = StructType([
    StructField("VendorID",             IntegerType()),
    StructField("tpep_pickup_datetime",  StringType()),
    StructField("tpep_dropoff_datetime", StringType()),
    StructField("passenger_count",       DoubleType()),
    StructField("trip_distance",         DoubleType()),
    StructField("RatecodeID",            DoubleType()),
    StructField("store_and_fwd_flag",    StringType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("extra",                 DoubleType()),
    StructField("mta_tax",               DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("tolls_amount",          DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount",          DoubleType()),
    StructField("congestion_surcharge",  DoubleType()),
    StructField("Airport_fee",           DoubleType()),
    StructField("cbd_congestion_fee",    DoubleType()),
])

# Read
bronze_df = spark.read.table("lakehouse.taxi.bronze")

# Parse JSON
parsed = (
    bronze_df
    .select(F.from_json(F.col("value"), trip_schema).alias("t"), "kafka_timestamp")
    .select(
        F.col("t.VendorID").alias("vendor_id"),
        F.to_timestamp("t.tpep_pickup_datetime").alias("pickup_datetime"),
        F.to_timestamp("t.tpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("t.passenger_count").cast(IntegerType()).alias("passenger_count"),
        F.col("t.trip_distance").alias("trip_distance"),
        F.col("t.RatecodeID").cast(IntegerType()).alias("rate_code_id"),
        F.col("t.store_and_fwd_flag").alias("store_and_fwd_flag"),
        F.col("t.PULocationID").alias("pu_location_id"),
        F.col("t.DOLocationID").alias("do_location_id"),
        F.col("t.payment_type").alias("payment_type"),
        F.col("t.fare_amount").alias("fare_amount"),
        F.col("t.extra").alias("extra"),
        F.col("t.mta_tax").alias("mta_tax"),
        F.col("t.tip_amount").alias("tip_amount"),
        F.col("t.tolls_amount").alias("tolls_amount"),
        F.col("t.improvement_surcharge").alias("improvement_surcharge"),
        F.col("t.total_amount").alias("total_amount"),
        F.col("t.congestion_surcharge").alias("congestion_surcharge"),
        F.col("t.Airport_fee").alias("airport_fee"),
        F.col("t.cbd_congestion_fee").alias("cbd_congestion_fee"),
        F.col("kafka_timestamp"),
    )
)

# Rules
cleaned = parsed.filter(
    F.col("vendor_id").isNotNull() &
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    F.col("pu_location_id").isNotNull() &
    F.col("do_location_id").isNotNull() &
    F.col("passenger_count").between(1, 6) &
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") > 0) &
    (F.col("total_amount") > 0) &
    (F.col("dropoff_datetime") > F.col("pickup_datetime"))
)

# Drop duplicates on these keys
deduped = cleaned.dropDuplicates([
    "vendor_id", "pickup_datetime", "pu_location_id", "do_location_id", "fare_amount"
])

# Enriching...
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")

pu_zones = zones.select(
    F.col("LocationID").alias("_pu_id"),
    F.col("Zone").alias("pickup_zone"),
    F.col("Borough").alias("pickup_borough"),
)
do_zones = zones.select(
    F.col("LocationID").alias("_do_id"),
    F.col("Zone").alias("dropoff_zone"),
    F.col("Borough").alias("dropoff_borough"),
)

silver_df = (
    deduped
    .join(pu_zones, deduped.pu_location_id == F.col("_pu_id"), "left").drop("_pu_id")
    .join(do_zones, deduped.do_location_id == F.col("_do_id"), "left").drop("_do_id")
)

# Write
silver_df.writeTo("lakehouse.taxi.silver").createOrReplace()
print("Silver write complete.")

# Validate
spark.sql("SELECT count(*) FROM lakehouse.taxi.silver").show()
spark.sql("SELECT vendor_id, pickup_datetime, pickup_zone, dropoff_zone, fare_amount, total_amount FROM lakehouse.taxi.silver LIMIT 5").show(truncate=False)